<a href="https://colab.research.google.com/github/Juno-Wong/Dome-41/blob/main/Dome_41.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Domesphere AI-powered Dashboard**

In [1]:
import pandas as pd

# Load all sheets
all_sheets = pd.read_excel("shameless_dashboard_200users.xlsx", sheet_name=None)

# Loop through each sheet and save as CSV
for sheet_name, df in all_sheets.items():
    df.to_csv(f"{sheet_name}.csv", index=False)

# **Clean the dataset**

## **1. Understand the structure**

In [2]:
df_users = pd.read_csv("shameless_users.csv")
df_posts = pd.read_csv("shameless_posts.csv")
df_comments = pd.read_csv("shameless_comments.csv")

In [3]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     200 non-null    object
 1   created_at  200 non-null    object
 2   username    200 non-null    object
 3   city        200 non-null    object
 4   industry    200 non-null    object
 5   gender      200 non-null    object
dtypes: object(6)
memory usage: 9.5+ KB


In [4]:
df_posts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   post_id              84 non-null     object
 1   user_id              84 non-null     object
 2   post_username        84 non-null     object
 3   post_created_at      84 non-null     object
 4   heading              84 non-null     object
 5   content              84 non-null     object
 6   embed_platform       44 non-null     object
 7   post_like_count      84 non-null     int64 
 8   post_like_usernames  66 non-null     object
 9   comment_count        84 non-null     int64 
 10  latest_comment_at    70 non-null     object
dtypes: int64(2), object(9)
memory usage: 7.3+ KB


In [5]:
df_comments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   comment_id           191 non-null    object
 1   post_id              191 non-null    object
 2   user_id              191 non-null    object
 3   comment_username     191 non-null    object
 4   comment_created_at   191 non-null    object
 5   content              191 non-null    object
 6   reply_to_comment_id  30 non-null     object
 7   comment_like_count   191 non-null    int64 
dtypes: int64(1), object(7)
memory usage: 12.1+ KB


## **2. Fix Data Types**

Ensures temporal consistency and enables downstream time-based analysis

In [6]:
# Users
df_users["created_at"] = pd.to_datetime(df_users["created_at"])

In [7]:
# Posts
df_posts["post_created_at"] = pd.to_datetime(df_posts["post_created_at"])
df_posts["latest_comment_at"] = pd.to_datetime(df_posts["latest_comment_at"])

In [8]:
# Comments
df_comments["comment_created_at"] = pd.to_datetime(df_comments["comment_created_at"])

### **3. Handle Missing Values (SMART handling only)**

**3.1 Posts**

In [9]:
df_posts["embed_platform"] = df_posts["embed_platform"].fillna("None")

In [10]:
df_posts["post_like_usernames"] = df_posts["post_like_usernames"].fillna("")

In [11]:
df_posts["latest_comment_at"] = df_posts["latest_comment_at"].fillna(df_posts["post_created_at"])

**3.2 Comments**

In [12]:
# Keep as is
# reply_to_comment_id stays NaN

Null values represent top-level comments (not replies).

*Explain: The reply_to_comment_id field contains null values for top-level comments, which do not reply to any existing comment. These null values were preserved, as they represent meaningful structural information rather than missing or erroneous data.*

### **4. Standardise Column Names**

In [13]:
df_users.rename(columns={"created_at": "user_created_at"})

,user_id,user_created_at,username,city,industry,gender
0,92a2012e-738d-4a74-becf-07dd07ab44f7,2026-01-07 08:36:00+00:00,lila,Wellington,Advertising,female
1,ec455a41-4c72-4c7d-9ace-89f42929f39e,2026-01-07 14:12:00+00:00,brookea,London,Student,female
2,03ec60cf-e03a-4715-87b7-3a25540371c5,2026-01-09 19:18:00+00:00,penny66,Bristol,Education,female
3,51efd9cc-9dd5-4483-b6de-ee8e728b82e9,2026-01-10 21:31:00+00:00,paige69,Brisbane,Healthcare,female
4,3c46fc4c-22a4-4dca-9078-1eb0d5d529cc,2026-01-12 09:48:00+00:00,mads,London,Legal,male
...,...,...,...,...,...,...
195,87f6beaf-55e7-47ea-80f0-70cda16730d2,2026-04-01 17:44:00+00:00,alixzz,Melbourne,Student,female
196,ec558623-0f9d-4d8b-88b5-51e03cea6aa4,2026-04-01 17:44:00+00:00,elsieclub,Auckland,Technology,female
197,a4dbd257-6e33-4ccb-bd1e-7ab23ad6c4c8,2026-04-02 21:18:00+00:00,jessie,Singapore,Education,female
198,27ce0655-01cf-4fa8-81e3-021f45b37a31,2026-04-04 07:37:00+00:00,bonnie_x,Christchurch,Advertising,female


Timestamp fields were preserved with distinct naming (e.g., `post_created_at`, `comment_created_at`, `user_created_at`) to maintain semantic clarity, as they represent different events within the system. While schema standardisation was considered, preserving meaningful distinctions was prioritised to support accurate downstream analysis.

### **5. Ensure Key Consistency**

Check relationships -> Remove orphan records and ensure relational integrity.

In [14]:
# Posts must have valid users
df_posts = df_posts[df_posts["user_id"].isin(df_users["user_id"])]

In [15]:
# Comments must have valid posts
df_comments = df_comments[df_comments["post_id"].isin(df_posts["post_id"])]

*Explain: Referential integrity was enforced by ensuring that all foreign key relationships were valid. Specifically, posts were filtered to include only records with `user_id` values present in the users dataset, and comments were filtered to include only valid `post_id` references. This prevents orphan records and ensures consistency across datasets.*

### **6. Remove Duplicates**

In [16]:
df_users = df_users.drop_duplicates()

In [17]:
df_posts = df_posts.drop_duplicates()

In [18]:
df_comments = df_comments.drop_duplicates()

Duplicate records were removed from all datasets to prevent redundancy and ensure data accuracy. This step helps avoid issues such as double counting and inconsistent analysis in downstream processes.

### **7. Clean Text**

In [19]:
import re

def basic_clean(text):
    text = str(text).strip()
    return text

df_posts["content"] = df_posts["content"].apply(basic_clean)
df_comments["content"] = df_comments["content"].apply(basic_clean)

Text fields were standardised by converting all values to string format and removing leading and trailing whitespace. This ensures consistency in textual data and prevents formatting-related issues in downstream processing.

### **8. Sort Data**

In [20]:
df_posts = df_posts.sort_values(by="post_created_at")
df_comments = df_comments.sort_values(by="comment_created_at")

###**9. Cleaned Output**

In [21]:
df_users.to_csv("clean_users.csv", index=False)
df_posts.to_csv("clean_posts.csv", index=False)
df_comments.to_csv("clean_comments.csv", index=False)

# **Feature Engineering**

In [22]:
import numpy as np
import re
from collections import Counter, defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer

###Load Cleaned Data

In [23]:
users = pd.read_csv("clean_users.csv")
posts = pd.read_csv("clean_posts.csv")
comments = pd.read_csv("clean_comments.csv")

#rename columns
if "created_at" in users.columns and "user_created_at" not in users.columns:
    users = users.rename(columns={"created_at": "user_created_at"})

###Convert date columns into datetime

In [24]:
users["user_created_at"] = pd.to_datetime(users["user_created_at"], errors="coerce")
posts["post_created_at"] = pd.to_datetime(posts["post_created_at"], errors="coerce")
posts["latest_comment_at"] = pd.to_datetime(posts["latest_comment_at"], errors="coerce")
comments["comment_created_at"] = pd.to_datetime(comments["comment_created_at"], errors="coerce")

###Assisting Functions - Clean text, Sentiment labels, Saving outputs and Clean text columns in posts and comments

In [28]:
def clean_text(text):
    """Basic text normalisation for NLP features."""
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

def save_feature(df, filename):
    df.to_csv(filename, index=False)
    print(f"Saved: {filename}")

posts["heading_clean"] = posts["heading"].fillna("").apply(clean_text)
posts["content_clean"] = posts["content"].fillna("").apply(clean_text)
posts["full_text"] = (posts["heading_clean"] + " " + posts["content_clean"]).str.strip()

comments["content_clean"] = comments["content"].fillna("").apply(clean_text)

###Building Event Tables

In [27]:
# Posts as events
post_events = posts[["user_id", "post_created_at"]].copy()
post_events["event_type"] = "post"
post_events = post_events.rename(columns={"post_created_at": "event_time"})

# Comments as events
comment_events = comments[["user_id", "comment_created_at"]].copy()
comment_events["event_type"] = "comment"
comment_events = comment_events.rename(columns={"comment_created_at": "event_time"})

# Combined interaction events
events = pd.concat([post_events, comment_events], ignore_index=True)
events["event_date"] = events["event_time"].dt.date
events["event_hour"] = events["event_time"].dt.hour
events["event_week"] = events["event_time"].dt.to_period("W").astype(str)
events["event_month"] = events["event_time"].dt.to_period("M").astype(str)

/tmp/ipykernel_1119/2787294732.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  events["event_week"] = events["event_time"].dt.to_period("W").astype(str)
/tmp/ipykernel_1119/2787294732.py:16: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  events["event_month"] = events["event_time"].dt.to_period("M").astype(str)


###Community Features

####Feature 1: Active Listeners Over Time

In [30]:
# Count distinct users per day from interaction records
active_listeners_daily = (
    events.groupby("event_date")["user_id"]
    .nunique()
    .reset_index(name="active_listeners")
)

active_listener_kpi = pd.DataFrame({
    "metric": ["total_active_days", "avg_daily_active_listeners", "max_daily_active_listeners"],
    "value": [
        active_listeners_daily["event_date"].nunique(),
        round(active_listeners_daily["active_listeners"].mean(), 2),
        active_listeners_daily["active_listeners"].max()
    ]
})

save_feature(active_listeners_daily, "feature_01_active_listeners_over_time.csv")
save_feature(active_listener_kpi, "feature_01_active_listeners_kpi.csv")

print("\n=== Feature 1: Active Listeners Over Time ===")
print(active_listeners_daily.head(10))
print("\n=== Feature 1 KPI ===")
print(active_listener_kpi)

Saved: feature_01_active_listeners_over_time.csv
Saved: feature_01_active_listeners_kpi.csv

=== Feature 1: Active Listeners Over Time ===
   event_date  active_listeners
0  2026-01-09                 2
1  2026-01-14                 2
2  2026-01-15                 2
3  2026-01-16                 2
4  2026-01-17                 4
5  2026-01-25                 5
6  2026-01-26                 4
7  2026-01-27                 1
8  2026-01-28                 3
9  2026-01-29                 2

=== Feature 1 KPI ===
                       metric  value
0           total_active_days  64.00
1  avg_daily_active_listeners   4.03
2  max_daily_active_listeners  11.00


####Feature 2: Audience Segmentation

In [31]:
# Count posts, comments, and received likes by city/gender/industry
post_counts_user = posts.groupby("user_id").size().reset_index(name="post_count")
comment_counts_user = comments.groupby("user_id").size().reset_index(name="comment_count")

# We only know like counts received, not who gave the likes.
post_likes_received = posts.groupby("user_id")["post_like_count"].sum().reset_index(name="post_likes_received")
comment_likes_received = comments.groupby("user_id")["comment_like_count"].sum().reset_index(name="comment_likes_received")

user_engagement = users[["user_id", "username", "city", "industry", "gender"]].copy()
for df in [post_counts_user, comment_counts_user, post_likes_received, comment_likes_received]:
    user_engagement = user_engagement.merge(df, on="user_id", how="left")

for col in ["post_count", "comment_count", "post_likes_received", "comment_likes_received"]:
    user_engagement[col] = user_engagement[col].fillna(0)

user_engagement["total_interactions"] = (
    user_engagement["post_count"] +
    user_engagement["comment_count"] +
    user_engagement["post_likes_received"] +
    user_engagement["comment_likes_received"]
)

audience_segment_city = (
    user_engagement.groupby("city")[["post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]]
    .sum()
    .reset_index()
    .sort_values("total_interactions", ascending=False)
)

audience_segment_gender = (
    user_engagement.groupby("gender")[["post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]]
    .sum()
    .reset_index()
    .sort_values("total_interactions", ascending=False)
)

audience_segment_industry = (
    user_engagement.groupby("industry")[["post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]]
    .sum()
    .reset_index()
    .sort_values("total_interactions", ascending=False)
)

save_feature(audience_segment_city, "feature_02_audience_segmentation_city.csv")
save_feature(audience_segment_gender, "feature_02_audience_segmentation_gender.csv")
save_feature(audience_segment_industry, "feature_02_audience_segmentation_industry.csv")

print("\n=== Feature 2: Audience Segmentation by City ===")
print(audience_segment_city.head(10))
print("\n=== Feature 2: Audience Segmentation by Gender ===")
print(audience_segment_gender)
print("\n=== Feature 2: Audience Segmentation by Industry ===")
print(audience_segment_industry.head(10))

Saved: feature_02_audience_segmentation_city.csv
Saved: feature_02_audience_segmentation_gender.csv
Saved: feature_02_audience_segmentation_industry.csv

=== Feature 2: Audience Segmentation by City ===
            city  post_count  comment_count  post_likes_received  \
8         London        16.0           24.0                 35.0   
15    Wellington         9.0           26.0                 14.0   
3        Bristol         9.0           24.0                 11.0   
1       Auckland        10.0           16.0                 18.0   
9     Manchester         6.0           13.0                  9.0   
6     Gold Coast         8.0           11.0                 17.0   
10     Melbourne         8.0           12.0                 11.0   
2       Brisbane         6.0           13.0                 13.0   
5   Christchurch         5.0            8.0                 13.0   
0       Adelaide         4.0           11.0                  4.0   

    comment_likes_received  total_interactions  

####Feature 3: Peak Activity Time

In [32]:
#Count interactions by hour using posts + comments
peak_activity_hour = (
    events.groupby("event_hour")
    .size()
    .reset_index(name="interaction_count")
    .sort_values("event_hour")
)

peak_hour = peak_activity_hour.loc[peak_activity_hour["interaction_count"].idxmax(), "event_hour"]
peak_hour_kpi = pd.DataFrame({
    "metric": ["peak_hour", "peak_hour_interactions"],
    "value": [int(peak_hour), int(peak_activity_hour["interaction_count"].max())]
})

save_feature(peak_activity_hour, "feature_03_peak_activity_time.csv")
save_feature(peak_hour_kpi, "feature_03_peak_activity_kpi.csv")

print("\n=== Feature 3: Peak Activity Time ===")
print(peak_activity_hour)
print("\n=== Feature 3 KPI ===")
print(peak_hour_kpi)

Saved: feature_03_peak_activity_time.csv
Saved: feature_03_peak_activity_kpi.csv

=== Feature 3: Peak Activity Time ===
    event_hour  interaction_count
0            0                  9
1            1                 11
2            2                  8
3            3                  7
4            4                  5
5            5                  7
6            6                  8
7            7                 12
8            8                 19
9            9                 21
10          10                 12
11          11                 24
12          12                 18
13          13                 15
14          14                 10
15          15                  7
16          16                  5
17          17                 11
18          18                 15
19          19                 12
20          20                 14
21          21                 10
22          22                  5
23          23                 10

=== Feature 3 KPI ===
       

####Feature 4: Top Contributors

In [33]:
# Count posts + comments + likes received per user
top_contributors = user_engagement[
    ["user_id", "username", "city", "industry", "gender",
     "post_count", "comment_count", "post_likes_received", "comment_likes_received", "total_interactions"]
].sort_values("total_interactions", ascending=False)

save_feature(top_contributors, "feature_04_top_contributors.csv")

print("\n=== Feature 4: Top Contributors ===")
print(top_contributors.head(20))

Saved: feature_04_top_contributors.csv

=== Feature 4: Top Contributors ===
                                 user_id    username          city  \
4   3c46fc4c-22a4-4dca-9078-1eb0d5d529cc        mads        London   
0   92a2012e-738d-4a74-becf-07dd07ab44f7        lila    Wellington   
72  1fd42c2d-fadb-4794-8d33-f4eefae244ac        cleo      Auckland   
8   8c3853ef-e9fd-489e-a00e-c4e6de6388c3        soph       Bristol   
92  f97fc0e9-dace-4428-8859-ad8ebd2532fd        tara     Newcastle   
15  7b71142d-776d-422c-9dd8-e8a1db4563e1       layla    Wellington   
28  c46eb7b5-1cd6-4b09-97c3-0cb70933c953        fern    Manchester   
91  0bf84a80-6729-4c12-80af-652ef37f5f0b        beth  Christchurch   
3   51efd9cc-9dd5-4483-b6de-ee8e728b82e9     paige69      Brisbane   
5   b3cbce25-bcfc-4039-8e81-460817f8514c  poppy_club        London   
1   ec455a41-4c72-4c7d-9ace-89f42929f39e     brookea        London   
30  971b74b9-34b6-4d68-b86b-6374e2cf764f       remis       Bristol   
93  17bb260a-3

###Content Features

In [34]:
#Topic Extraction Function
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=100,
    ngram_range=(1, 2),
    min_df=1
)

post_tfidf = vectorizer.fit_transform(posts["full_text"])
feature_names = np.array(vectorizer.get_feature_names_out())

# Dominant topic for each post
top_term_idx = np.asarray(post_tfidf.argmax(axis=1)).ravel()
posts["dominant_topic"] = feature_names[top_term_idx]

####Feature 5: Trending Topics

In [42]:
# Most discussed keywords/topics in posts and comments

combined_text = posts["full_text"].fillna("")

custom_stopwords = [
    "think", "people", "just", "want", "really", "actually", "interesting",
    "love", "best", "good", "great", "dont", "didnt", "doesnt", "feel",
    "feels", "going", "doing", "coming", "need", "read", "know", "like",
    "episode", "podcast", "online", "fully", "angle", "pr"
]

topic_vectorizer = TfidfVectorizer(
    stop_words=custom_stopwords,
    max_features=50,
    ngram_range=(2, 8),   # focus on recurring phrases instead of single words
    min_df=2
)

topic_matrix = topic_vectorizer.fit_transform(combined_text)
topic_scores = np.asarray(topic_matrix.sum(axis=0)).ravel()
topic_terms = topic_vectorizer.get_feature_names_out()

trending_topics = pd.DataFrame({
    "topic_phrase": topic_terms,
    "score": topic_scores
}).sort_values("score", ascending=False).reset_index(drop=True)

save_feature(trending_topics, "feature_05_trending_topics.csv")

print("\n=== Feature 5: Recurring Topics / Phrases ===")
print(trending_topics.head(20))

Saved: feature_05_trending_topics.csv

=== Feature 5: Recurring Topics / Phrases ===
                                    topic_phrase      score
0                                       than the  20.762479
1                                        for the  14.603023
2                                        what is   6.751297
3                                      mind over   4.281723
4                     whether changed their mind   4.281723
5                whether changed their mind over   4.281723
6            whether changed their mind over the   4.281723
7                             the last few weeks   4.281723
8   whether changed their mind over the last few   4.281723
9                          whether changed their   4.281723
10                               whether changed   4.281723
11      whether changed their mind over the last   4.281723
12                                  the last few   4.281723
13                              please send your   3.495644
14            m

####Feature 6: Trending Conversations

In [43]:
# Rank posts by comments + likes
trending_conversations = posts[
    ["post_id", "post_username", "post_created_at", "heading", "post_like_count", "comment_count", "dominant_topic"]
].copy()

trending_conversations["conversation_score"] = (
    trending_conversations["post_like_count"] + trending_conversations["comment_count"]
)

trending_conversations = trending_conversations.sort_values(
    "conversation_score", ascending=False
).reset_index(drop=True)

save_feature(trending_conversations, "feature_06_trending_conversations.csv")

print("\n=== Feature 6: Trending Conversations ===")
print(trending_conversations.head(20))

Saved: feature_06_trending_conversations.csv

=== Feature 6: Trending Conversations ===
                                 post_id post_username  \
0   e1d99ba5-08bb-4cb6-86b0-9b5aaae3ee28          beth   
1   c09b7467-d5f1-456c-8f6e-5d015f0248b7    poppy_club   
2   60165a64-6147-4751-a15a-89cc89847ccf         elsie   
3   96893c7b-7acc-4d2e-820f-a2977f4e8af8        nell35   
4   a5805a92-36f4-4cc4-b616-30b62410a9f6       josie75   
5   d9bb1bd3-1106-4db0-9c63-2968dd55af46       hallezz   
6   59a7b8cf-33d1-4d84-855f-e02b9045476f           mae   
7   45917656-6d24-4051-b55f-c6590833f526          beth   
8   f5f35d98-c72f-43fe-aac3-c33bde9ce310          mads   
9   c9f7127c-9264-451c-b158-7b0ccf7aff0b          mads   
10  0e3815a8-14f2-4da9-bac0-254a9b9149de          lila   
11  1b3e2721-f5d7-434b-a363-69c6a7f2a42c        isla_x   
12  e1edcf7b-0b8b-40b8-b643-d60f9af9051e          lila   
13  86e188be-c8ac-4809-8637-6528b92f210b       layla_x   
14  4ce780f8-fac3-49d1-b04a-6157c0daaac6  

####Feature 7: Sentiment Analysis

In [46]:
#VADER - sentiment analysis tool (Positive, Negative, Neutral)
try:
    from nltk.sentiment import SentimentIntensityAnalyzer
    import nltk

    try:
        nltk.data.find("sentiment/vader_lexicon.zip")
    except LookupError:
        nltk.download("vader_lexicon")

    USE_VADER = True
except:
    USE_VADER = False

# Apply sentiment scoring
if USE_VADER:
    sia = SentimentIntensityAnalyzer()

    posts["sentiment_score"] = posts["full_text"].apply(
        lambda x: sia.polarity_scores(str(x))["compound"]
    )
    comments["sentiment_score"] = comments["content_clean"].apply(
        lambda x: sia.polarity_scores(str(x))["compound"]
    )

else:
    positive_words = {
        "love", "great", "good", "amazing", "fun", "helpful",
        "interesting", "best", "excited", "happy"
    }
    negative_words = {
        "bad", "boring", "hate", "annoying", "worst", "confusing",
        "poor", "hard", "sad", "frustrating"
    }

    def simple_sentiment(text):
        words = set(str(text).lower().split())
        pos = len(words & positive_words)
        neg = len(words & negative_words)

        if pos + neg == 0:
            return 0.0
        return (pos - neg) / (pos + neg)

    posts["sentiment_score"] = posts["full_text"].apply(simple_sentiment)
    comments["sentiment_score"] = comments["content_clean"].apply(simple_sentiment)

# Convert score into label
def sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

posts["sentiment_label"] = posts["sentiment_score"].apply(sentiment_label)
comments["sentiment_label"] = comments["sentiment_score"].apply(sentiment_label)

# Summaries
sentiment_summary_posts = (
    posts.groupby("sentiment_label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

sentiment_summary_comments = (
    comments.groupby("sentiment_label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

save_feature(sentiment_summary_posts, "feature_07_sentiment_posts.csv")
save_feature(sentiment_summary_comments, "feature_07_sentiment_comments.csv")

print("\n=== Feature 7: Sentiment Summary (Posts) ===")
print(sentiment_summary_posts)

print("\n=== Feature 7: Sentiment Summary (Comments) ===")
print(sentiment_summary_comments)

Saved: feature_07_sentiment_posts.csv
Saved: feature_07_sentiment_comments.csv

=== Feature 7: Sentiment Summary (Posts) ===
  sentiment_label  count
1        positive     76
0        negative      8

=== Feature 7: Sentiment Summary (Comments) ===
  sentiment_label  count
2        positive     94
1         neutral     63
0        negative     34


####Feature 8: Content Performance by Topic

In [47]:
# Group posts by dominant topic
content_performance_by_topic = (
    posts.groupby("dominant_topic")
    .agg(
        post_count=("post_id", "count"),
        avg_post_likes=("post_like_count", "mean"),
        avg_comments=("comment_count", "mean"),
        total_post_likes=("post_like_count", "sum"),
        total_comments=("comment_count", "sum")
    )
    .reset_index()
)

content_performance_by_topic["engagement_score"] = (
    content_performance_by_topic["avg_post_likes"] +
    content_performance_by_topic["avg_comments"]
)

content_performance_by_topic = content_performance_by_topic.sort_values(
    "engagement_score", ascending=False
)

save_feature(content_performance_by_topic, "feature_08_content_performance_by_topic.csv")

print("\n=== Feature 8: Content Performance by Topic ===")
print(content_performance_by_topic.head(20))


Saved: feature_08_content_performance_by_topic.csv

=== Feature 8: Content Performance by Topic ===
          dominant_topic  post_count  avg_post_likes  avg_comments  \
6                changed           1        3.000000      3.000000   
19            reality tv           8        2.750000      2.625000   
18                  read           2        2.500000      2.500000   
16                  like           1        3.000000      2.000000   
7               circling           3        1.000000      3.666667   
10             discourse           8        2.000000      2.625000   
15                 group           7        2.285714      2.285714   
20              smartest           2        3.500000      1.000000   
14               fatigue           6        1.666667      2.500000   
2                  angle           6        1.833333      2.333333   
17                  open          11        1.818182      2.272727   
8                  click           1        1.000000      3.

###Commercial Features

####Feature 9: Engagement Rate

In [49]:
posts_daily = posts.groupby(posts["post_created_at"].dt.date).size().reset_index(name="posts")
comments_daily = comments.groupby(comments["comment_created_at"].dt.date).size().reset_index(name="comments")
post_likes_daily = posts.groupby(posts["post_created_at"].dt.date)["post_like_count"].sum().reset_index(name="post_likes")
comment_likes_daily = comments.groupby(comments["comment_created_at"].dt.date)["comment_like_count"].sum().reset_index(name="comment_likes")

engagement_rate = posts_daily.merge(
    comments_daily,
    left_on="post_created_at",
    right_on="comment_created_at",
    how="outer"
)

engagement_rate = engagement_rate.rename(columns={"post_created_at": "date"}).drop(
    columns=["comment_created_at"], errors="ignore"
)

engagement_rate = engagement_rate.merge(
    post_likes_daily.rename(columns={"post_created_at": "date"}),
    on="date",
    how="outer"
)

engagement_rate = engagement_rate.merge(
    comment_likes_daily.rename(columns={"comment_created_at": "date"}),
    on="date",
    how="outer"
)

# Fill only numeric columns, not the date column
for col in ["posts", "comments", "post_likes", "comment_likes"]:
    engagement_rate[col] = engagement_rate[col].fillna(0)

engagement_rate = engagement_rate.sort_values("date")

engagement_rate["total_interactions"] = (
    engagement_rate["posts"] +
    engagement_rate["comments"] +
    engagement_rate["post_likes"] +
    engagement_rate["comment_likes"]
)

total_members = users["user_id"].nunique()
engagement_rate["total_members"] = total_members
engagement_rate["engagement_rate_pct"] = (
    engagement_rate["total_interactions"] / engagement_rate["total_members"]
) * 100

save_feature(engagement_rate, "feature_09_engagement_rate.csv")

print("\n=== Feature 9: Engagement Rate ===")
print(engagement_rate.head(20))

Saved: feature_09_engagement_rate.csv

=== Feature 9: Engagement Rate ===
          date  posts  comments  post_likes  comment_likes  \
0   2026-01-09    1.0       1.0         2.0            3.0   
1   2026-01-14    1.0       1.0         4.0            1.0   
2   2026-01-15    0.0       0.0         0.0            3.0   
3   2026-01-16    1.0       1.0         3.0            3.0   
4   2026-01-17    1.0       5.0         3.0            2.0   
5   2026-01-25    2.0       3.0         3.0            3.0   
6   2026-01-26    1.0       4.0         4.0            2.0   
7   2026-01-27    1.0       0.0         2.0            0.0   
8   2026-01-28    2.0       2.0         1.0            1.0   
9   2026-01-29    0.0       0.0         0.0            4.0   
10  2026-01-30    0.0       0.0         0.0            2.0   
11  2026-01-31    3.0       1.0         6.0            1.0   
12  2026-02-01    1.0       5.0         2.0            3.0   
13  2026-02-02    1.0       2.0         4.0            4.0